# Cross-dataset comparison
Reads only the `summary.json` of each dataset repository, plus the pHash tables for overlap.

Code: GPL-3.0-only, Copyright (C) 2026 Mohammad Hossein Omoumi (https://github.com/omoumi/skin-metadata-audit). The dataset keeps its own license.

In [ ]:
# install the toolkit: the attached Kaggle copy, otherwise the tagged GitHub release
import glob, os, shutil, subprocess, sys
REPO, TAG = "https://github.com/omoumi/skin-metadata-audit.git", "v0.1.0"
!pip install -q imagehash==4.3.2 scikit-learn==1.6.1
TOOLKIT = "/kaggle/temp/skin-metadata-audit"
local = sorted(glob.glob("/kaggle/input/**/skin-metadata-audit/pyproject.toml", recursive=True), key=len)
if local:
    shutil.copytree(os.path.dirname(local[0]), TOOLKIT, dirs_exist_ok=True)
elif not os.path.exists(os.path.join(TOOLKIT, "pyproject.toml")):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", TAG, REPO, TOOLKIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", TOOLKIT], check=True)
import skinaudit
print("skinaudit", skinaudit.__version__, "installed from", os.path.dirname(local[0]) if local else f"{REPO}@{TAG}")

In [ ]:
import glob, os, pandas as pd, numpy as np
from skinaudit.compare import build_comparison
# one summary per dataset (sub-analyses such as hiba_audit/dermoscopic_only left out)
found = sorted(glob.glob("/kaggle/input/**/summary.json", recursive=True))
summaries = [p for p in found if not os.path.exists(os.path.join(os.path.dirname(os.path.dirname(p)), "summary.json"))]
print(len(summaries), "datasets:", *summaries, sep="\n")
OUT = "/kaggle/working/comparison"
res = build_comparison(summaries, OUT)
res["rubric"]

In [ ]:
# shared images between datasets: same image id, and pHash candidate pairs within 4 bits
from skinaudit.dedup import load_hashes, overlap_table
tabs = {p.split("/")[-3]: load_hashes(p) for p in sorted(glob.glob("/kaggle/input/**/tables/phash.csv", recursive=True))
        if os.path.dirname(os.path.dirname(p)) in {os.path.dirname(s) for s in summaries}}
overlap, pairs = overlap_table(tabs, max_dist=4)
overlap.to_csv(f"{OUT}/dataset_overlap.csv", index=False)
pairs.to_csv(f"{OUT}/dataset_overlap_pairs.csv.gz", index=False)
overlap.sort_values(["shared_ids", "phash_pairs"], ascending=False)

In [ ]:
# Zip the output folder for download
import shutil
print(shutil.make_archive(OUT, "zip", os.path.dirname(OUT), os.path.basename(OUT)))